# NLP project

In [1]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


In [2]:
from llama_cpp import Llama

In [3]:
# Load the model
llm = Llama.from_pretrained(repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", # repository name
                            filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf", # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

In [22]:
import requests
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/ticket_to_ride.txt').text

In [23]:
rulebook[:100]

'# components\n- 1 rules booklet\n- 1 game board (a map of North American train routes)\n- Plastic train'

In [46]:
prompts = ["""You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation""",
           """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the to a child by comparing it to something they already know (e.g., “like a treasure hunt” or “like building a LEGO city”).  Use the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""
]
game_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid']

In [47]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [49]:
from tqdm import tqdm
outputs = {}
for filename,prompt in tqdm([(f,p) for f in game_names for p in prompts]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+filename+'.txt').text
    outputs[filename+prompt] = llm.create_chat_completion(
        messages= generate_message(prompt, rulebook),
        temperature=0.7,
    )['choices'][0]['message']['content']

100%|██████████| 8/8 [02:15<00:00, 16.94s/it]


In [50]:
for o in outputs:
    print(outputs[o])

Hey there, young adventurer! Let's talk about the exciting game of Train Tracks!

**The Goal:**
The goal of the game is to score the most points by claiming routes on the map, completing tickets, and building the longest continuous path of trains.

**How to Win:**
You win the game by having the most points at the end. To get points, you need to:

* Claim routes on the map by playing train cards
* Complete tickets you kept by linking the cities on the ticket with your trains
* Build the longest continuous path of trains

**What's a Turn:**
On your turn, you can do one of three things:

1. **Draw Train Cards:** Take 2 train cards from the deck or the face-up cards on the table.
2. **Claim a Route:** Choose a route on the map and play the right number of train cards to claim it.
3. **Draw Tickets:** Take 3 tickets from the deck and keep at least one.

**Important Rules:**

* You can only claim one route per turn.
* You can only discard tickets you just drew, not the ones you already had.
